# 🏗️ Spanner & BigQuery Knowledge Graph Demo with Document AI & Reverse ETL

This notebook demonstrates an end-to-end operational and analytical Knowledge Graph architecture across **Spanner**, **BigQuery**, **Document AI**, and **Gemini**:
* **Transactional Storage (Spanner)**: Stores core relational business entities (`Customers`, `Products`, `PurchaseOrders`, `CustomerComplaints`) and runs operational GQL graph queries.
* **Document Parsing (Document AI)**: Automatically extracts text structure from PDF technical manuals stored in GCS.
* **AI Extraction (BigQuery & Gemini)**: Extracts entities (Parts, Materials) and relationships (`CONTAINS_PART`, `MADE_OF`) using Gemini `AI.GENERATE`.
* **Hybrid Property Graph (BigQuery)**: Joins Spanner transactional data (via BigQuery Federated Queries) with extracted Document AI entities into a Property Graph.
* **Reverse ETL (BigQuery -> Spanner)**: Computes high-risk component intelligence from the Graph and pushes enriched insights back into Spanner.

All modular SQL scripts for Spanner schema, extraction, graph building, visualization, and Reverse ETL are stored as separate `.sql` files in `./sql/` and can be executed independently.

## 🏁 Phase 1: Environment Setup & Constants

In [ ]:
!pip install google-cloud-documentai google-cloud-bigquery google-cloud-spanner --quiet

In [ ]:
import os
from google.cloud import bigquery, storage, documentai
from google.api_core.client_options import ClientOptions

# Update these values to match your GCP project environment
PROJECT_ID = "your-project-id" # @param {type:"string"}
LOCATION = 'us' # @param {type:"string"}
DATASET_ID = 'kg_demo' # @param {type:"string"}
SPANNER_INSTANCE_ID = 'kg-spanner-instance' # @param {type:"string"}
SPANNER_DATABASE_ID = 'kg-spanner-db' # @param {type:"string"}
SPANNER_CONNECTION_ID = f"{LOCATION}.spanner_kg_conn" # BigQuery Connection to Spanner

GCS_FILE_LOCATION = f"gs://{PROJECT_ID}-kg-demo-data"
GEMINI_MODEL_VERSION = f"https://aiplatform.googleapis.com/v1/projects/{PROJECT_ID}/locations/global/publishers/google/models/gemini-3.1-pro-preview"

OBJECT_TABLE_NAME = "manufacturing_manuals"
GRAPH_TABLE_NAME = "manufacturing_kg"
EXTRACTED_KG_TABLE_NAME = "extracted_knowledge_graph"
PROCESSED_DOCUMENTS_TABLE = "processed_documents"

### 📄 Setup Document AI Processor

In [ ]:
# Setup Document AI Client & Layout Parser Processor
processor_display_name = "pdf_parser_processor"
opts = ClientOptions(api_endpoint=f"{LOCATION}-documentai.googleapis.com")
client = documentai.DocumentProcessorServiceClient(client_options=opts)
parent = client.common_location_path(PROJECT_ID, LOCATION)

existing_processors = client.list_processors(parent=parent)
processor = next((p for p in existing_processors if p.display_name == processor_display_name), None)

if processor:
    print(f"Found existing processor: {processor.name}")
else:
    print("Creating new LAYOUT_PARSER_PROCESSOR...")
    processor = client.create_processor(
        parent=parent,
        processor=documentai.Processor(
            display_name=processor_display_name,
            type_="LAYOUT_PARSER_PROCESSOR"
        ),
    )

PROCESSOR_NAME = processor.name
print(f"Processor Name: {PROCESSOR_NAME}")

## Phase 2: Document Ingestion and Parsing with Document AI
Parses technical manual PDFs stored in Google Cloud Storage using BigQuery `AI.PARSE_DOCUMENT`.

In [ ]:
bq_client = bigquery.Client(project=PROJECT_ID)

parse_sql = f"""
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.{DATASET_ID}.{OBJECT_TABLE_NAME}`
WITH CONNECTION DEFAULT
OPTIONS(
 object_metadata = 'SIMPLE',
 uris = ['{GCS_FILE_LOCATION}/*.pdf']);

CREATE OR REPLACE TABLE `{DATASET_ID}.{PROCESSED_DOCUMENTS_TABLE}` AS (
 SELECT * FROM AI.PARSE_DOCUMENT(
   TABLE `{DATASET_ID}.{OBJECT_TABLE_NAME}`,
   endpoint => "{PROCESSOR_NAME}",
   chunk_size => 250
 )
);
"""
print("Executing Document AI parsing...")
bq_client.query(parse_sql).result()
print("Document AI parsing complete.")

## Phase 3: AI Knowledge Extraction (`sql/06_bq_extract_knowledge.sql`)
Executes `AI.GENERATE` via external SQL script to extract structured nodes and edges from document chunks.

In [ ]:
def run_sql_file(filepath: str, replacements: dict):
    with open(filepath, 'r') as f:
        sql = f.read()
    for k, v in replacements.items():
        sql = sql.replace(f"{{{k}}}", str(v))
    return sql

extract_sql = run_sql_file('sql/06_bq_extract_knowledge.sql', {
    'DATASET_ID': DATASET_ID,
    'EXTRACTED_KG_TABLE_NAME': EXTRACTED_KG_TABLE_NAME,
    'PROCESSED_DOCUMENTS_TABLE': PROCESSED_DOCUMENTS_TABLE,
    'GEMINI_MODEL_VERSION': GEMINI_MODEL_VERSION
})

print("Executing Knowledge Extraction SQL file...")
bq_client.query(extract_sql).result()
print("Extraction complete.")

## Phase 4: Graph Construction & Spanner Integration
Executes `sql/05_bq_spanner_federated_views.sql` and `sql/07_bq_build_graph.sql` to build BigQuery property graph with Spanner federated transactional tables.

In [ ]:
# 1. Create Federated Views over Spanner
fed_sql = run_sql_file('sql/05_bq_spanner_federated_views.sql', {
    'DATASET_ID': DATASET_ID,
    'SPANNER_CONNECTION_ID': SPANNER_CONNECTION_ID
})
bq_client.query(fed_sql).result()

# 2. Build Node, Edge tables and BigQuery Property Graph
graph_sql = run_sql_file('sql/07_bq_build_graph.sql', {
    'DATASET_ID': DATASET_ID,
    'EXTRACTED_KG_TABLE_NAME': EXTRACTED_KG_TABLE_NAME,
    'GRAPH_TABLE_NAME': GRAPH_TABLE_NAME
})
bq_client.query(graph_sql).result()
print("BigQuery Property Graph constructed successfully.")

## Phase 5: Reverse ETL Execution (`sql/12_bq_reverse_etl_to_spanner.sql`)
Computes complaint component vulnerabilities in BigQuery analytics and synchronizes insights back to Spanner.

In [ ]:
reverse_etl_sql = run_sql_file('sql/12_bq_reverse_etl_to_spanner.sql', {
    'DATASET_ID': DATASET_ID,
    'SPANNER_CONNECTION_ID': SPANNER_CONNECTION_ID
})
bq_client.query(reverse_etl_sql).result()
print("Reverse ETL pipeline complete.")

## 🎨 Phase 6: Visualizations & Graph Queries
Renders graph visualisations by executing SQL query files in `sql/`.

### Viz 1: Full Graph Traversal (`sql/08_bq_viz_full_graph.sql`)

In [ ]:
v1_sql = run_sql_file('sql/08_bq_viz_full_graph.sql', {
    'DATASET_ID': DATASET_ID,
    'GRAPH_TABLE_NAME': GRAPH_TABLE_NAME
})
df1 = bq_client.query(v1_sql).to_dataframe()
df1.head()

### Viz 2: Product -> Part -> Material Subgraph (`sql/09_bq_viz_product_materials.sql`)

In [ ]:
v2_sql = run_sql_file('sql/09_bq_viz_product_materials.sql', {
    'DATASET_ID': DATASET_ID,
    'GRAPH_TABLE_NAME': GRAPH_TABLE_NAME
})
df2 = bq_client.query(v2_sql).to_dataframe()
df2.head()

### Viz 3: Customer Purchases with Fiberglass Components (`sql/10_bq_viz_customer_fiberglass.sql`)

In [ ]:
v3_sql = run_sql_file('sql/10_bq_viz_customer_fiberglass.sql', {
    'DATASET_ID': DATASET_ID,
    'GRAPH_TABLE_NAME': GRAPH_TABLE_NAME
})
df3 = bq_client.query(v3_sql).to_dataframe()
df3.head()

### Viz 4: Customer Complaints & Material Traceability (`sql/11_bq_viz_customer_complaints.sql`)

In [ ]:
v4_sql = run_sql_file('sql/11_bq_viz_customer_complaints.sql', {
    'DATASET_ID': DATASET_ID,
    'GRAPH_TABLE_NAME': GRAPH_TABLE_NAME
})
df4 = bq_client.query(v4_sql).to_dataframe()
df4.head()